# Lab 10 Student Version (Google Colab)
## Apply Mixed Precision and Gradient Clipping on BERT (DistilBERT)

This notebook is a **student-learning version** of Lab 10. It follows **every step and topic** from the lab guide and adds short explanations of **what you are learning** and **why each step matters**.

## Lab overview
In this lab, you will improve the performance and stability of a BERT-based sentiment classifier by adding two widely used optimization techniques:
- **Automatic Mixed Precision (AMP)**: speeds up training and reduces GPU memory usage by using float16 where safe and float32 where needed.
- **Gradient clipping**: prevents unstable training by limiting gradient magnitude (helps avoid exploding gradients).

You will:
- load and preprocess a small IMDb subset (1000 rows),
- tokenize with DistilBERT tokenizer,
- build a PyTorch Dataset + DataLoader pipeline,
- train with a **manual training loop** that integrates AMP + gradient clipping,
- evaluate validation accuracy,
- and complete a **mini challenge** to time and compare standard vs AMP training.

### Estimated completion time
- **35 minutes**

### Runtime type (important)
- Set Runtime type to: **T4 GPU**


# Task 1: Preparing the IMDb dataset and model

### What you are learning
You are learning how to set up a Colab environment for transformer training, then prepare a small IMDb dataset subset for fast fine-tuning:
- install Hugging Face libraries (`transformers`, `datasets`)
- switch to GPU for faster training
- upload the dataset CSV
- shuffle + label encode + limit to 1000 rows
- tokenize with DistilBERT tokenizer
- wrap the data in a PyTorch Dataset class and split into train/validation

---

## Step 1: Install dependencies (run once)


In [ ]:
!pip install -q transformers datasets


## Step 2–3: Switch runtime to T4 GPU and restart session

### What you are learning
AMP is most beneficial on GPU. The lab requires:
- **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**
- **Runtime → Restart session**

After restarting, re-run the notebook from the top.


## Step 4: Import required libraries

### What you are learning
These imports cover:
- data processing (pandas)
- dataset batching (DataLoader, Dataset)
- tokenizer + model (transformers)
- optimizer (AdamW)
- evaluation (accuracy_score)
- splitting (train_test_split)
- progress bars (tqdm)


In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm


## Step 5–6: Upload the IMDb dataset CSV

### What you are learning
In Colab, your local lab files are not present by default, so you upload them.

**Upload file:** `IMDB Dataset.csv`


In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))


## Step 7: Read, shuffle, label-encode, and limit to 1000 rows

### What you are learning
- Shuffling prevents ordering effects.
- Mapping labels to 0/1 makes the dataset numeric for training.
- Limiting to **1000 rows** keeps training fast for a lab.


In [ ]:
df = pd.read_csv("IMDB Dataset.csv")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})
df = df[["review", "label"]]
df = df[:1000]

print("Rows used:", len(df))
display(df.head(3))
print("Label counts:\n", df["label"].value_counts())


## Step 8: Tokenize and create a Dataset class

### What you are learning
Transformers require tokenized inputs:
- `input_ids` (token IDs)
- `attention_mask` (1 for real tokens, 0 for padding)

You will:
1) load the DistilBERT tokenizer  
2) define a PyTorch Dataset that tokenizes in `__init__()`  
3) split the data into training and validation sets (80/20)


In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

class IMDBDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True, max_length=256
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["review"], df["label"], test_size=0.2, random_state=42
)

train_dataset = IMDBDataset(train_texts.tolist(), train_labels.tolist())
val_dataset   = IMDBDataset(val_texts.tolist(),   val_labels.tolist())

print("Train samples:", len(train_dataset))
print("Val samples:  ", len(val_dataset))
print("Example keys:", train_dataset[0].keys())


### Task 1 takeaways
- You now have a tokenized dataset pipeline compatible with PyTorch DataLoaders and a transformer model.


# Task 2: Adding mixed precision and gradient clipping to the training loop

### What you are learning
You will train DistilBERT using a **manual training loop** and integrate:
- **AMP autocast**: runs certain operations in float16 for speed
- **GradScaler**: prevents underflow by scaling losses/gradients
- **Gradient clipping**: caps gradient norm (`max_norm=1.0`) to stabilize training

---

## Step 1: Load pretrained model, move to device, and create optimizer


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)

print("Using device:", device)
print("Model loaded with num_labels=2")


## Step 2: Prepare DataLoaders

### What you are learning
DataLoaders create mini-batches:
- training uses `shuffle=True`
- validation uses default (no shuffle)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8)

print("Train batches:", len(train_loader))
print("Val batches:  ", len(val_loader))


## Step 3: Manual training loop with AMP + gradient clipping

### What you are learning
This loop adds three key steps:
1) `autocast(...)` for mixed precision forward pass  
2) `GradScaler` to safely backprop scaled gradients  
3) `clip_grad_norm_` to prevent exploding gradients  

> The lab uses `from torch.amp import autocast, GradScaler`.  
> Some environments use `torch.cuda.amp`. The code below supports both.


In [ ]:
# AMP imports (compatible across PyTorch versions)
try:
    from torch.amp import autocast, GradScaler
    _amp_device = "cuda" if device.type == "cuda" else "cpu"
    scaler = GradScaler(_amp_device)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()

epochs = 1

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        # Mixed precision forward pass
        if device.type == "cuda":
            with autocast("cuda"):
                outputs = model(**batch)
                loss = outputs.loss
        else:
            # CPU fallback (AMP benefit is mainly on GPU)
            outputs = model(**batch)
            loss = outputs.loss

        # Scaled backward pass
        scaler.scale(loss).backward()

        # Unscale before clipping so clipping uses real gradient values
        scaler.unscale_(optimizer)

        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Step optimizer with scaler
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")


### Task 2 takeaways
- AMP speeds up training on GPU and reduces memory usage.
- Gradient clipping can prevent instability when gradients become too large.
- Using a manual loop teaches you exactly where AMP and clipping fit into training.


# Task 3: Evaluating and comparing performance

### What you are learning
You will evaluate validation performance using **accuracy**:
- switch to `model.eval()`
- disable gradients with `torch.no_grad()`
- convert logits to predicted classes with `argmax`
- compare predictions to true labels to compute accuracy


In [ ]:
model.eval()
predictions, true_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(batch["labels"].cpu().numpy())

acc = accuracy_score(true_labels, predictions)
print(f"\nValidation Accuracy: {acc * 100:.2f}%")


# Mini challenge: Timing comparison (Lab 9 standard vs Lab 10 AMP + clipping)

### What you are learning
You will empirically compare training time using Python’s timing utility:
- **Standard training** (no AMP, no gradient clipping)  
vs  
- **AMP + gradient clipping** (Lab 10 approach)

### Requirements (from the lab)
Use identical settings in both experiments:
- same dataset (1000 IMDb samples)
- same model architecture (DistilBERT)
- same number of epochs and batch size

> Note: This notebook provides a *standard manual loop* to represent “Lab 9 standard training,”
> and an AMP+clipping loop to represent “Lab 10 training.”


In [ ]:
import time

def standard_train_one_epoch():
    model_std = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    ).to(device)
    opt_std = AdamW(model_std.parameters(), lr=5e-5)
    model_std.train()

    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        opt_std.zero_grad()
        outputs = model_std(**batch)
        loss = outputs.loss
        loss.backward()
        opt_std.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def amp_clip_train_one_epoch():
    model_amp = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    ).to(device)
    opt_amp = AdamW(model_amp.parameters(), lr=5e-5)
    model_amp.train()

    # AMP scaler
    try:
        from torch.amp import autocast as _autocast, GradScaler as _GradScaler
        _amp_device = "cuda" if device.type == "cuda" else "cpu"
        _scaler = _GradScaler(_amp_device)
    except Exception:
        from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
        _scaler = _GradScaler()

    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        opt_amp.zero_grad()

        if device.type == "cuda":
            with _autocast("cuda"):
                outputs = model_amp(**batch)
                loss = outputs.loss
        else:
            outputs = model_amp(**batch)
            loss = outputs.loss

        _scaler.scale(loss).backward()
        _scaler.unscale_(opt_amp)
        torch.nn.utils.clip_grad_norm_(model_amp.parameters(), max_norm=1.0)
        _scaler.step(opt_amp)
        _scaler.update()

        total_loss += loss.item()
    return total_loss / len(train_loader)

# Time standard training
start_time = time.time()
std_loss = standard_train_one_epoch()
end_time = time.time()
print(f"[Standard] Avg Loss: {std_loss:.4f} | Training Time: {end_time - start_time:.2f} seconds")

# Time AMP + clipping training
start_time = time.time()
amp_loss = amp_clip_train_one_epoch()
end_time = time.time()
print(f"[AMP+Clip] Avg Loss: {amp_loss:.4f} | Training Time: {end_time - start_time:.2f} seconds")


# Lab review

1. What does mixed precision training aim to optimize?

A. Model accuracy  
B. GPU memory usage  
C. Training speed and efficiency  
D. Tokenization time  

---

## STOP
You have successfully completed this lab.


<details>
<summary><strong>Optional self-check answer (click to expand)</strong></summary>

**C. Training speed and efficiency** (and often reduced GPU memory usage).

</details>
